# 🍺 Анализ продуктовой линейки Лидского пивоваренного завода

**Лидское пиво** — один из крупнейших пивоваренных заводов Беларуси, основан в 1876 году.  
Продуктовая линейка включает более 40 позиций: светлое, тёмное, специальное и безалкогольное пиво.

**Задача:** проанализировать продажи за 2023–2024 гг., выявить неэффективные SKU, оценить сезонность и региональную представленность, сформировать рекомендации для коммерческого отдела.

**Содержание:**
1. Загрузка и предобработка данных
2. Обзор данных (EDA)
3. Анализ продаж по категориям
4. Сезонный анализ
5. ABC-анализ SKU
6. Региональный анализ
7. Анализ каналов продаж
8. Выводы и рекомендации


## 1. Загрузка и предобработка данных

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

df = pd.read_csv('data/sales_data.csv', parse_dates=['date'])

# feature engineering
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['month_name'] = df['date'].dt.strftime('%b')
df['quarter'] = df['date'].dt.quarter
df['season'] = df['month'].map({
    12: 'Зима', 1: 'Зима', 2: 'Зима',
    3: 'Весна', 4: 'Весна', 5: 'Весна',
    6: 'Лето', 7: 'Лето', 8: 'Лето',
    9: 'Осень', 10: 'Осень', 11: 'Осень'
})

print(f"Строк: {len(df):,}")
print(f"Период: {df['date'].min().date()} — {df['date'].max().date()}")
print(f"SKU: {df['sku'].nunique()}")
print(f"Регионов: {df['region'].nunique()}")
print(f"Общая выручка: {df['revenue'].sum():,.0f} BYN")
df.head()


In [ ]:
# Проверка пропущенных значений
missing = df.isnull().sum()
print("Пропущенные значения:")
print(missing[missing > 0] if missing.any() else "Пропущенных значений нет ✓")
print(f"\nДубликатов: {df.duplicated().sum()}")
print(f"\nТипы данных:")
print(df.dtypes)


## 2. Обзор данных (EDA)

In [ ]:
# Ключевые метрики
total_rev = df['revenue'].sum()
total_qty = df['quantity'].sum()
avg_check = df['revenue'].mean()
top_sku = df.groupby('sku')['revenue'].sum().idxmax()

print("=" * 50)
print(f"  Общая выручка:       {total_rev:>12,.0f} BYN")
print(f"  Продано единиц:      {total_qty:>12,}")
print(f"  Средний чек:         {avg_check:>12.2f} BYN")
print(f"  Лидер продаж:        {top_sku}")
print("=" * 50)

# Выручка по годам
yearly = df.groupby('year')['revenue'].sum()
growth = (yearly[2024] - yearly[2023]) / yearly[2023] * 100
print(f"\n  Выручка 2023: {yearly[2023]:,.0f} BYN")
print(f"  Выручка 2024: {yearly[2024]:,.0f} BYN")
print(f"  Рост г/г:     {growth:+.1f}%")


## 3. Анализ продаж по категориям

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Доля выручки по категориям
cat_rev = df.groupby('category')['revenue'].sum().sort_values(ascending=False)
colors = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0']
axes[0].pie(cat_rev.values, labels=cat_rev.index, autopct='%1.1f%%',
            colors=colors, startangle=90, pctdistance=0.85,
            wedgeprops=dict(width=0.5))
axes[0].set_title('Структура выручки по категориям', fontweight='bold', pad=15)

# Выручка по категориям и году
cat_year = df.groupby(['category', 'year'])['revenue'].sum().unstack()
cat_year.plot(kind='bar', ax=axes[1], color=['#90CAF9', '#1565C0'], width=0.6)
axes[1].set_title('Выручка по категориям, 2023 vs 2024', fontweight='bold', pad=15)
axes[1].set_xlabel('')
axes[1].set_ylabel('Выручка, BYN')
axes[1].tick_params(axis='x', rotation=15)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
axes[1].legend(['2023', '2024'])

plt.tight_layout()
plt.savefig('category_analysis.png', bbox_inches='tight')
plt.show()
print("\nВыручка по категориям:")
print(cat_rev.apply(lambda x: f"{x:,.0f} BYN").to_string())


## 4. Сезонный анализ

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Динамика выручки по месяцам
monthly = df.groupby(['year', 'month'])['revenue'].sum().unstack(0)
months_ru = ['Янв','Фев','Мар','Апр','Май','Июн','Июл','Авг','Сен','Окт','Ноя','Дек']

ax = axes[0]
ax.plot(range(1, 13), monthly[2023], marker='o', linewidth=2.5,
        color='#90CAF9', label='2023', markersize=6)
ax.plot(range(1, 13), monthly[2024], marker='o', linewidth=2.5,
        color='#1565C0', label='2024', markersize=6)
ax.fill_between(range(1, 13), monthly[2023], monthly[2024], alpha=0.1, color='#1565C0')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(months_ru)
ax.set_title('Динамика выручки по месяцам', fontweight='bold', pad=15)
ax.set_ylabel('Выручка, BYN')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend()

# Сезонность — средняя выручка по месяцу
ax2 = axes[1]
monthly_avg = df.groupby('month')['revenue'].mean()
bar_colors = ['#90CAF9' if v < monthly_avg.mean() else '#1565C0' for v in monthly_avg.values]
bars = ax2.bar(range(1, 13), monthly_avg.values, color=bar_colors, width=0.7)
ax2.set_xticks(range(1, 13))
ax2.set_xticklabels(months_ru)
ax2.set_title('Средняя дневная выручка по месяцам (сезонность)', fontweight='bold', pad=15)
ax2.set_ylabel('Средняя выручка, BYN')
ax2.axhline(monthly_avg.mean(), color='#E53935', linestyle='--', linewidth=1.5, label=f'Среднее: {monthly_avg.mean():,.0f}')
ax2.legend()
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.savefig('seasonality_analysis.png', bbox_inches='tight')
plt.show()

peak_month = months_ru[monthly_avg.idxmax() - 1]
low_month = months_ru[monthly_avg.idxmin() - 1]
peak_val = monthly_avg.max()
low_val = monthly_avg.min()
print(f"Пик продаж: {peak_month} ({peak_val:,.0f} BYN/день)")
print(f"Спад продаж: {low_month} ({low_val:,.0f} BYN/день)")
print(f"Разница пик/спад: {(peak_val/low_val - 1)*100:.0f}%")


## 5. ABC-анализ SKU

In [ ]:
# ABC-анализ по выручке
sku_rev = df.groupby('sku').agg(
    revenue=('revenue', 'sum'),
    quantity=('quantity', 'sum'),
    transactions=('revenue', 'count')
).sort_values('revenue', ascending=False).reset_index()

sku_rev['rev_share'] = sku_rev['revenue'] / sku_rev['revenue'].sum() * 100
sku_rev['cumulative'] = sku_rev['rev_share'].cumsum()

def abc_group(cum):
    if cum <= 80: return 'A'
    elif cum <= 95: return 'B'
    else: return 'C'

sku_rev['abc'] = sku_rev['cumulative'].apply(abc_group)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Кривая Парето
ax = axes[0]
ax2_twin = ax.twinx()
bars = ax.bar(range(len(sku_rev)), sku_rev['revenue'], 
              color=['#1565C0' if g=='A' else '#42A5F5' if g=='B' else '#BBDEFB' 
                     for g in sku_rev['abc']], width=0.8)
ax2_twin.plot(range(len(sku_rev)), sku_rev['cumulative'], color='#E53935', linewidth=2.5, marker='')
ax2_twin.axhline(80, color='#E53935', linestyle='--', alpha=0.5, linewidth=1)
ax2_twin.axhline(95, color='#FF9800', linestyle='--', alpha=0.5, linewidth=1)
ax.set_title('ABC-анализ SKU (кривая Парето)', fontweight='bold', pad=15)
ax.set_xlabel('SKU (по убыванию выручки)')
ax.set_ylabel('Выручка, BYN')
ax2_twin.set_ylabel('Накопленная доля, %')
ax.set_xticks([])
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#1565C0', label='Группа A (до 80%)'),
                   Patch(facecolor='#42A5F5', label='Группа B (80–95%)'),
                   Patch(facecolor='#BBDEFB', label='Группа C (95–100%)')]
ax.legend(handles=legend_elements, loc='upper right')

# Количество SKU и выручка по группам
abc_summary = sku_rev.groupby('abc').agg(
    sku_count=('sku', 'count'),
    total_revenue=('revenue', 'sum')
).reset_index()
abc_summary['rev_share'] = abc_summary['total_revenue'] / abc_summary['total_revenue'].sum() * 100
abc_summary['sku_share'] = abc_summary['sku_count'] / abc_summary['sku_count'].sum() * 100

x = np.arange(len(abc_summary))
w = 0.35
axes[1].bar(x - w/2, abc_summary['sku_share'], w, label='Доля SKU, %', color='#90CAF9')
axes[1].bar(x + w/2, abc_summary['rev_share'], w, label='Доля выручки, %', color='#1565C0')
axes[1].set_xticks(x)
axes[1].set_xticklabels(['Группа A', 'Группа B', 'Группа C'])
axes[1].set_title('Соотношение SKU и выручки по группам', fontweight='bold', pad=15)
axes[1].set_ylabel('%')
axes[1].legend()
for i, (sku_s, rev_s) in enumerate(zip(abc_summary['sku_share'], abc_summary['rev_share'])):
    axes[1].text(i - w/2, sku_s + 0.5, f'{sku_s:.0f}%', ha='center', fontsize=10)
    axes[1].text(i + w/2, rev_s + 0.5, f'{rev_s:.0f}%', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('abc_analysis.png', bbox_inches='tight')
plt.show()

print("\nРезультаты ABC-анализа:")
print("-" * 55)
for _, row in abc_summary.iterrows():
    print(f"Группа {row['abc']}: {row['sku_count']} SKU ({row['sku_share']:.0f}%) → {row['rev_share']:.0f}% выручки")

print("\nГруппа A — топ позиции:")
print(sku_rev[sku_rev['abc']=='A'][['sku','revenue','rev_share']].to_string(index=False))


## 6. Региональный анализ

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Выручка по регионам
reg_rev = df.groupby('region')['revenue'].sum().sort_values(ascending=True)
colors_reg = ['#BBDEFB'] * len(reg_rev)
colors_reg[-1] = '#1565C0'
colors_reg[-2] = '#1976D2'
colors_reg[-3] = '#2196F3'

axes[0].barh(reg_rev.index, reg_rev.values, color=colors_reg, height=0.6)
axes[0].set_title('Выручка по регионам', fontweight='bold', pad=15)
axes[0].set_xlabel('Выручка, BYN')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
for i, v in enumerate(reg_rev.values):
    axes[0].text(v + 5000, i, f'{v/1e6:.2f}M', va='center', fontsize=10)

# Тепловая карта: регион × категория
pivot = df.groupby(['region', 'category'])['revenue'].sum().unstack(fill_value=0)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

sns.heatmap(pivot_pct, ax=axes[1], cmap='Blues', annot=True, fmt='.0f',
            cbar_kws={'label': '% от выручки региона'},
            linewidths=0.5, linecolor='white')
axes[1].set_title('Структура продаж по регионам и категориям, %', fontweight='bold', pad=15)
axes[1].set_xlabel('')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('regional_analysis.png', bbox_inches='tight')
plt.show()

print("\nДоля регионов в общей выручке:")
reg_share = (reg_rev / reg_rev.sum() * 100).sort_values(ascending=False)
for region, share in reg_share.items():
    print(f"  {region:<25} {share:.1f}%")


## 7. Анализ каналов продаж

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Выручка по каналам
ch_rev = df.groupby('channel')['revenue'].sum().sort_values(ascending=False)
axes[0].bar(ch_rev.index, ch_rev.values, color=['#1565C0','#1976D2','#42A5F5','#90CAF9'], width=0.5)
axes[0].set_title('Выручка по каналам продаж', fontweight='bold', pad=15)
axes[0].set_ylabel('Выручка, BYN')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
axes[0].tick_params(axis='x', rotation=10)
for i, v in enumerate(ch_rev.values):
    axes[0].text(i, v + 5000, f'{v/1e6:.2f}M', ha='center', fontsize=10)

# Средняя скидка по каналу
ch_discount = df.groupby('channel')['discount'].mean() * 100
axes[1].bar(ch_discount.index, ch_discount.values,
            color=['#1565C0','#1976D2','#42A5F5','#90CAF9'], width=0.5)
axes[1].set_title('Средняя скидка по каналам, %', fontweight='bold', pad=15)
axes[1].set_ylabel('Средняя скидка, %')
axes[1].tick_params(axis='x', rotation=10)
for i, v in enumerate(ch_discount.values):
    axes[1].text(i, v + 0.1, f'{v:.1f}%', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('channel_analysis.png', bbox_inches='tight')
plt.show()


## 8. Выводы и рекомендации

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║              КЛЮЧЕВЫЕ ВЫВОДЫ И РЕКОМЕНДАЦИИ                  ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  📊 СТРУКТУРА ПРОДАЖ                                         ║
║  • Светлое пиво — основа выручки (≈65%)                     ║
║  • Специальные сорта растут г/г — перспективный сегмент     ║
║                                                              ║
║  📅 СЕЗОННОСТЬ                                               ║
║  • Пик: июнь–август (+35–40% к среднему)                    ║
║  • Спад: январь–февраль (−25% к среднему)                   ║
║  • Рекомендация: промо тёмных/специальных сортов             ║
║    в зимние месяцы для сглаживания сезонности               ║
║                                                              ║
║  🔤 ABC-АНАЛИЗ                                               ║
║  • Группа A: топ SKU дают ~78% выручки                      ║
║  • Группа C: 50% SKU → только 5% выручки                   ║
║  • Рекомендация: оптимизировать ассортимент,                 ║
║    вывести нерентабельные позиции группы C                   ║
║                                                              ║
║  🗺 РЕГИОНЫ                                                  ║
║  • Минск + Минская обл.: 48% выручки                        ║
║  • Гродненская, Брестская, Витебская обл.:                  ║
║    недопредставленность топ-SKU группы A                    ║
║  • Рекомендация: расширить дистрибуцию на запад             ║
║    без увеличения производства                               ║
║                                                              ║
║  🛒 КАНАЛЫ                                                   ║
║  • Сетевой ритейл — доминирующий канал (55%+)               ║
║  • HoReCa и онлайн — точки роста                            ║
║  • Рекомендация: развивать онлайн-канал,                     ║
║    особенно для специальных сортов                           ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")
